In [1]:
import fitz                # pip install PyMuPDF
import unicodedata
import re
import pandas as pd

In [2]:
def strip_accents(text: str) -> str:
    # elimina marcas diacríticas (tildes)
    text = unicodedata.normalize('NFD', text)
    return ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')

In [3]:
def extract_blocks(pdf_path: str) -> list[str]:
    doc = fitz.open(pdf_path)
    all_text = ""
    for page in doc:
        all_text += page.get_text("text") + "\n\f\n"
    # normalizamos y partimos en cada ficha
    norm = strip_accents(all_text)
    parts = norm.split("FICHA TECNICA")
    # el primer fragmento antes de la primera ficha suele estar vacío o contener cabeceras
    return parts[1:]

In [4]:
def parse_block(block: str) -> dict:
    lines = [ln.strip() for ln in block.splitlines()]
    # helper para la siguiente línea con contenido
    def next_value(i):
        for j in range(i+1, len(lines)):
            if lines[j]:
                return lines[j]
        return ""

    data = {}
    for i, ln in enumerate(lines):
        if ln == "No. CAMARA":
            val = next_value(i)
            m = re.match(r"(\d+)/(\d{4})([CS])", val)
            if m:
                data["num_camara"], data["anio_camara"], letra = m.groups()
                data["origen_camara"] = "Camara" if letra == "C" else "Senado"
        elif ln == "No. SENADO":
            val = next_value(i)
            m = re.match(r"(\d+)/(\d{4})([CS])", val)
            if m:
                data["num_senado"], data["anio_senado"], letra = m.groups()
                data["origen_senado"] = "Camara" if letra == "C" else "Senado"
        elif ln == "RADICACION":
            data["fecha_radicacion"] = next_value(i)
        elif ln == "TIPO DE PROYECTO":
            data["tipo_proyecto"] = next_value(i)
        elif ln == "SEUDONIMO":
            data["seudonimo"] = next_value(i)
        elif ln == "COMISION":
            data["comision"] = next_value(i)
        elif ln == "CAMARA DE ORIGEN":
            data["camara_origen"] = next_value(i)
        elif ln == "TITULO":
            # acumula hasta el próximo encabezado (AUTOR o PONENTES)
            tit = []
            for j in range(i+1, len(lines)):
                if lines[j].startswith("AUTOR") or lines[j].startswith("Ponentes") or not lines[j]:
                    break
                tit.append(lines[j])
            data["titulo"] = " ".join(tit)
        elif ln.startswith("AUTOR"):
            data["autores"] = next_value(i)
        # Debates Vuelta 1
        elif ln.startswith("Ponentes Primer Debate Camara"):
            data["1v_ponentes1_cam"] = next_value(i)
        elif ln.startswith("Ponentes Segundo Debate Camara"):
            data["1v_ponentes2_cam"] = next_value(i)
        elif ln.startswith("Ponentes Primer Debate Senado"):
            data["1v_ponentes1_sen"] = next_value(i)
        elif ln.startswith("Ponentes Segundo Debate Senado"):
            data["1v_ponentes2_sen"] = next_value(i)
        # Publicaciones vuelta 1
        elif ln == "PUBLICACIONES GACETA":
            data["1v_pub_gaceta"] = next_value(i)
        elif ln == "CAMARA DE REPRESENTANTES":
            data["1v_pub_camara"] = next_value(i)
        elif ln == "SENADO DE LA REPUBLICA":
            data["1v_pub_senado"] = next_value(i)
        # Miembros conciliación 1
        elif ln.startswith("Miembros comision de conciliacion Camara"):
            data["1v_miembros_cam"] = next_value(i)
        elif ln.startswith("Miembros comision de conciliacion Senado"):
            data["1v_miembros_sen"] = next_value(i)
        # Actos legislativos y Observaciones vuelta 1
        elif ln.startswith("Acto Legislativo"):
            # puede haber dos actos; acumulamos todos
            acts = re.findall(r"No\.?\s*(\d+)", block)
            data["1v_actos"] = ", ".join(acts)
        elif ln.startswith("Observaciones"):
            data["1v_observaciones"] = next_value(i)
        # Segunda vuelta: igual, prefijando con "2v_"
        elif ln.startswith("SEGUNDA VUELTA"):
            sec = lines[i+1:]
            # aquí podrías repetir la misma lógica de debate/publicaciones usando
            # el mismo helper sobre `sec`, o bien continuar sobre `lines` detectando
            # etiquetas con prefijo "2v_"
            # (Para no alargar demasiado, lo dejo pendiente de tu formato exacto)
            pass
        elif ln == "ESTADO ACTUAL":
            data["estado_actual"] = " ".join(lines[i+1:]).strip()
    return data

In [5]:
def main(pdf_path: str, excel_path: str):
    bloques = extract_blocks(pdf_path)
    registros = [parse_block(b) for b in bloques]
    # Para verificar que haya levantado algo:
    print(f"Encontradas {len(registros)} fichas con datos (mirar DataFrame).")
    df = pd.DataFrame(registros)
    df.to_excel(excel_path, index=False)
    print("Excel generado en:", excel_path)

In [6]:
if __name__ == "__main__":
    import sys
    pdf_path   = r"C:\Users\juans\Documents\proarchitecg\Model-Extract-information\extract\modelos\team\2022 2023 LEGISLATURA_proyectos_ley_actos_Legislativos.pdf"
    excel_path = "fichas_tecnicas.xlsx"
    main(pdf_path, excel_path)

Encontradas 425 fichas con datos (mirar DataFrame).
Excel generado en: fichas_tecnicas.xlsx
